### Import libraries

In [1]:
import pandas as pd
import numpy as np
import os
import math
import seaborn as sns
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import GofChisquarePower
from sklearn.cluster import KMeans
import sklearn.metrics as sil
from ratio import find_power_vals

### Import humans

In [2]:
humans = pd.read_pickle('./data/phenotype/humans_maf_popstrat.pkl')

### Looping

In [3]:
maf_pops = []
for f in list(humans.columns):
    try:
        f.split("pheno_maf_")[1]
        maf_pops.append(f.split("pheno_maf_")[1])
    except:
        pass

In [4]:
df_chrom = []
df_file_name = []
df_snp_name = []
df_maf = []
df_maf_pop = []
df_intercept = []
df_effect = []
df_p_intercept = []
df_p_effect = []
df_SE = []
df_power = []

In [15]:
list(set(df_tot['file_name']))

['5_2_1200_5_maf_0.073_0.1.pkl',
 '3_8_923_3_maf_0.035_0.051.pkl',
 '0_9_800_0_maf_0.01_0.016.pkl',
 '6_3_800_6_maf_0.1_0.13.pkl',
 '8_0_980_8_maf_0.17_0.21.pkl',
 '11_6_800_11_maf_0.32_0.38.pkl',
 '11_0_1200_11_maf_0.32_0.38.pkl',
 '7_0_1200_7_maf_0.13_0.17.pkl',
 '0_1_1200_0_maf_0.01_0.016.pkl',
 '8_5_800_8_maf_0.17_0.21.pkl',
 '4_9_1200_4_maf_0.051_0.073.pkl',
 '3_2_1077_3_maf_0.035_0.051.pkl',
 '9_7_800_9_maf_0.21_0.27.pkl',
 '4_7_800_4_maf_0.051_0.073.pkl',
 '10_2_1200_10_maf_0.27_0.32.pkl',
 '1_0_1200_1_maf_0.016_0.023.pkl',
 '7_5_800_7_maf_0.13_0.17.pkl',
 '2_5_800_2_maf_0.023_0.035.pkl',
 '3_7_800_3_maf_0.035_0.051.pkl',
 '2_7_800_2_maf_0.023_0.035.pkl',
 '12_7_800_12_maf_0.38_0.44.pkl',
 '1_7_1200_1_maf_0.016_0.023.pkl',
 '2_8_1200_2_maf_0.023_0.035.pkl',
 '10_1_1116_10_maf_0.27_0.32.pkl',
 '9_0_1200_9_maf_0.21_0.27.pkl',
 '4_5_1200_4_maf_0.051_0.073.pkl',
 '4_4_800_4_maf_0.051_0.073.pkl',
 '2_9_800_2_maf_0.023_0.035.pkl',
 '1_8_830_1_maf_0.016_0.023.pkl',
 '2_4_800_2_maf_0.02

In [5]:
chroms = [21,22]
for chrom in chroms:
    for snps in os.listdir(f"./data/genotype/chrom_{chrom}"):
        path_snp = f"./data/genotype/chrom_{chrom}/{snps}"
        geno = pd.read_pickle(path_snp)
        for i in range(len(geno.columns)):
            snp_num = i
            snp_name = geno.columns[snp_num]
            geno[geno.columns[snp_num]].value_counts()
            num_maj = geno[geno.columns[snp_num]].value_counts()[1]
            num_het = geno[geno.columns[snp_num]].value_counts()[0]
            try:
                num_min = geno[geno.columns[snp_num]].value_counts()[-1]
            except Exception as e:
                num_min = 0
            total_humans = num_maj + num_het + num_min
            maf = (num_min*2 + num_het)/(total_humans*2)
            min_power = 80
            [a, b, c] = find_power_vals(min_power)
            ratio = a*maf**2 + b*maf + c

            true_mean = 0
            true_effect = 5
            true_sd = true_effect*ratio

            mean_min = true_mean
            mean_het = true_mean
            mean_maj = true_mean
            sigma_min = true_sd
            sigma_het = true_sd
            sigma_maj = true_sd

            for maf_pop in maf_pops:
                humans[f"causal"] = geno[geno.columns[snp_num]]
                humans['pheno'] = humans[f"pheno_maf_{maf_pop}"]
                X = humans[['causal']]
                y = humans[['pheno']]
                reg = LinearRegression().fit(X, y)
                estimated_effect = reg.coef_[0][0]
                estimated_intercept = reg.intercept_[0]
                N = len(X)
                p = len(X.columns) + 1

                X_with_intercept = np.empty(shape=(N, p), dtype=float)
                X_with_intercept[:, 0] = 1
                X_with_intercept[:, 1:p] = X.values

                params = np.append(reg.intercept_,reg.coef_)
                predictions = reg.predict(X)
                new_y = (y-predictions)**2
                newX = pd.DataFrame({"Constant":np.ones(len(X))}).join(pd.DataFrame(list(X['causal'])))
                MSE = (sum(list(new_y['pheno'])))/(len(newX)-len(newX.columns))

                var_b = MSE*(np.linalg.inv(np.dot(newX.T,newX)).diagonal())
                sd_b = np.sqrt(var_b)
                ts_b = params/ sd_b
                p_values =[2*(1-stats.t.cdf(np.abs(i),(len(newX)-len(newX.columns)))) for i in ts_b]

                residuals = y - predictions
                residual_sum_of_squares = residuals.T @ residuals
                sigma_squared_hat = residual_sum_of_squares['pheno'].pheno / (N - p)
                var_beta_hat = np.linalg.inv(X_with_intercept.T @ X_with_intercept) * sigma_squared_hat
                SE = []
                for p_ in range(p):
                    standard_error = var_beta_hat[p_, p_] ** 0.5
                    SE.append(standard_error)

                sd_b = np.round(sd_b,3)
                ts_b = np.round(ts_b,3)
                params = np.round(params,4)

                distribution_nul = np.random.normal(0, SE[1], 5000)
                distribution_alt = np.random.normal(estimated_effect, SE[1], 5000)

                dis_only_alt = pd.DataFrame(data={"Z": list(distribution_alt)})
                dis_only_alt['Z2'] = (dis_only_alt['Z']/SE[1])**2
                cuttof = stats.chi2.ppf(0.8, df=1)
                omega = 0
                for val in list(dis_only_alt['Z2']):
                    if val > cuttof:
                        omega += 1
                    else:
                        pass
                power = omega/len(list(dis_only_alt['Z2']))
                #print(f"chrom = {chrom}")
                #print(f"file name = {snps}")
                #print(f"snp name = {snp_name}")
                #print(f"maf = {maf}")
                #print(f"strat distribution = {maf_pop}")
                #print(f"estimated intercept = {estimated_intercept}")
                #print(f"estimated effect = {estimated_effect}")
                #print(f"p value intercept = {p_values[0]}")
                #print(f"p value effect = {p_values[1]}")
                #print(f"standard error = {SE[1]}")
                #print(f"power = {round(power*100,8)}")
                
                df_chrom.append(chrom)
                df_file_name.append(snps)
                df_snp_name.append(snp_name)
                df_maf.append(maf)
                df_maf_pop.append(maf_pop)
                df_intercept.append(estimated_intercept)
                df_effect.append(estimated_effect)
                df_p_intercept.append(p_values[0])
                df_p_effect.append(p_values[1])
                df_SE.append(SE[1])
                df_power.append(power)
df_tot = pd.DataFrame(data = {'chrom':df_chrom,
                              'file_name':df_file_name,
                              'snp_name': df_snp_name,
                              'maf':df_maf,
                              'maf_pop': df_maf_pop,
                              'intercept':df_intercept,
                              'effect':df_effect,
                              'p_intercept': df_p_intercept,
                              'p_effect': df_p_effect,
                              'SE': df_SE,
                              'power':df_power
                             })
df_tot.to_pickle("./data/effect_pop_strat2.pkl")

/tmp/ipykernel_877/2671470145.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  humans[f"causal"] = geno[geno.columns[snp_num]]
/tmp/ipykernel_877/2671470145.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  humans['pheno'] = humans[f"pheno_maf_{maf_pop}"]


KeyboardInterrupt: 

In [ ]:
df_tot = pd.read_pickle("./data/effect_pop_strat.pkl")
df_tot

In [6]:
df_tot = pd.DataFrame(data = {'chrom':df_chrom,
                              'file_name':df_file_name,
                              'snp_name': df_snp_name,
                              'maf':df_maf,
                              'maf_pop': df_maf_pop,
                              'intercept':df_intercept,
                              'effect':df_effect,
                              'p_intercept': df_p_intercept,
                              'p_effect': df_p_effect,
                              'SE': df_SE,
                              'power':df_power
                             })
df_tot.to_pickle("./data/effect_pop_strat.pkl")

In [9]:
df_tot.sort_values(by=['effect'])

,chrom,file_name,snp_name,maf,maf_pop,intercept,effect,p_intercept,p_effect,SE,power
5459726,21,3_9_1200_3_maf_0.035_0.051.pkl,21:10802538:A:G_G,0.047724,0.32_6,2.551582,-2.940503,0.0,0.0,0.115066,1.0
5459717,21,3_9_1200_3_maf_0.035_0.051.pkl,21:10802538:A:G_G,0.047724,0.26_7,2.405381,-2.928655,0.0,0.0,0.115065,1.0
5459737,21,3_9_1200_3_maf_0.035_0.051.pkl,21:10802538:A:G_G,0.047724,0.38_7,2.469623,-2.923595,0.0,0.0,0.105459,1.0
5459689,21,3_9_1200_3_maf_0.035_0.051.pkl,21:10802538:A:G_G,0.047724,0.12_9,2.257439,-2.910873,0.0,0.0,0.098212,1.0
191049,21,0_8_800_0_maf_0.01_0.016.pkl,21:34161973:C:T_C,0.013578,0.12_9,2.448500,-2.902932,0.0,0.0,0.198042,1.0
...,...,...,...,...,...,...,...,...,...,...,...
5488449,21,3_9_1200_3_maf_0.035_0.051.pkl,21:27301160:A:G_A,0.042332,0.26_9,-2.274824,2.933309,0.0,0.0,0.117084,1.0
2951109,21,2_9_800_2_maf_0.023_0.035.pkl,21:36358323:A:C_C,0.024361,0.26_9,-2.404753,2.959062,0.0,0.0,0.158957,1.0
1063639,21,4_9_1200_4_maf_0.051_0.073.pkl,21:29087229:G:T_T,0.053714,0.26_9,-2.251193,2.981643,0.0,0.0,0.100674,1.0
886452,21,1_8_830_1_maf_0.016_0.023.pkl,21:31401646:A:G_G,0.015775,0.32_2,-2.689224,3.049643,0.0,0.0,0.222294,1.0


In [11]:
df_tot['effectsquared'] = df_tot['effect']**2

In [12]:
df_tot.sort_values(by=['effectsquared'])

,chrom,file_name,snp_name,maf,maf_pop,intercept,effect,p_intercept,p_effect,SE,power,effectsquared
4494529,21,11_4_1200_11_maf_0.32_0.38.pkl,21:21964257:A:G_A,0.380391,0.035_9,-0.017676,-4.344586e-09,1.373177e-02,1.000000,0.009701,0.2058,1.887543e-17
4370500,21,11_4_1200_11_maf_0.32_0.38.pkl,21:42904315:C:T_C,0.365615,0.035_0,0.154319,-1.432280e-08,0.000000e+00,0.999999,0.012070,0.1936,2.051427e-16
2233765,21,10_0_986_10_maf_0.27_0.32.pkl,21:20136216:A:G_A,0.317093,0.26_5,-0.323607,3.447862e-08,0.000000e+00,0.999999,0.047565,0.2028,1.188775e-15
6198461,21,7_2_800_7_maf_0.13_0.17.pkl,21:20205451:A:G_A,0.136182,0.091_1,0.226125,-3.462597e-08,5.138556e-12,0.999999,0.036740,0.2090,1.198958e-15
6283275,21,2_8_1200_2_maf_0.023_0.035.pkl,21:26097931:A:C_C,0.026558,0.32_5,0.111962,5.433636e-08,3.895620e-01,1.000000,0.133422,0.1960,2.952440e-15
...,...,...,...,...,...,...,...,...,...,...,...,...
5459726,21,3_9_1200_3_maf_0.035_0.051.pkl,21:10802538:A:G_G,0.047724,0.32_6,2.551582,-2.940503e+00,0.000000e+00,0.000000,0.115066,1.0000,8.646555e+00
2951109,21,2_9_800_2_maf_0.023_0.035.pkl,21:36358323:A:C_C,0.024361,0.26_9,-2.404753,2.959062e+00,0.000000e+00,0.000000,0.158957,1.0000,8.756050e+00
1063639,21,4_9_1200_4_maf_0.051_0.073.pkl,21:29087229:G:T_T,0.053714,0.26_9,-2.251193,2.981643e+00,0.000000e+00,0.000000,0.100674,1.0000,8.890194e+00
886452,21,1_8_830_1_maf_0.016_0.023.pkl,21:31401646:A:G_G,0.015775,0.32_2,-2.689224,3.049643e+00,0.000000e+00,0.000000,0.222294,1.0000,9.300325e+00
